# User Guide

This guide is a thorough tour of the Euromod Connector. It assumes you have
already installed the package and can load a model (if not, start with
[Getting Started](getstarted.ipynb)). Where Getting Started shows *the happy
path*, this guide explains *the object model* and the full set of options for
running simulations and building counterfactual reforms.

**Contents**

1. [Setup and reproducible paths](#setup)
2. [The `Model` object](#model)
3. [Navigating the model: countries, the object hierarchy](#navigate)
4. [The spine: policies, functions, parameters](#spine)
5. [Systems and datasets](#systems)
6. [Running a simulation](#running)
7. [Building reforms I — policy switches](#switches)
8. [Building reforms II — overwriting constants](#constants)
9. [Add-ons](#addons)
10. [Extensions](#extensions)
11. [Requesting specific outputs](#requested)
12. [Run options reference](#options)
13. [Working across countries](#multicountry)
14. [Caveats](#caveats)

(setup)=
## 1. Setup and reproducible paths

A simulation needs a **model** (policy rules) and **input microdata**. Throughout
this guide we use the **training datasets** that ship with every public EUROMOD
release, so nothing here depends on confidential data.

Edit the two paths below to point at your own EUROMOD installation. `MODEL_PATH`
is the model/release folder; `DATA_DIR` is where the input microdata lives (the
model's `Input` folder in a standard release).

In [1]:
import os
import pandas as pd
from euromod import Model

# --- Point these to your local EUROMOD installation -------------------------
# MODEL_PATH : a EUROMOD model/release folder (the one that contains 'XMLParam').
# DATA_DIR   : folder with the input microdata (.txt). In a public release this
#              is the model's own 'Input' folder, which ships with the freely
#              distributable *training* datasets used throughout these docs.
MODEL_PATH = r"C:\EUROMOD\EUROMOD_RELEASE"
DATA_DIR   = os.path.join(MODEL_PATH, "Input")

# keep DataFrame previews compact in the rendered docs
pd.set_option("display.max_rows", 8, "display.max_columns", 8)

Using EUROMOD as defined in C:\EUROMOD\Executable


(model)=
## 2. The `Model` object

`Model` is the entry point. Pass it the path to a EUROMOD model folder (the one
containing `XMLParam`). Loading is lazy where it can be, so creating the object
is cheap.

In [3]:
mod = Model(MODEL_PATH)
mod

------------------------------
Model
------------------------------
	 countries: 28 elements
	 extensions: 17 elements
	 model_path: 'C:\\EUROMOD\\EUROMOD_RELEASE'

Every connector object has an **informative representation**: printing it shows
its user-relevant attributes and how many elements each container holds. The
`Model` has two you will use constantly:

- `countries` — a container of `Country` objects;
- `extensions` — the model-wide extensions (more on these
  [below](#extensions)).

In [4]:
mod.extensions

0: Benefit Take-up Adjustments           
1: Tax Compliance Adjustments            
2: Full Year Adjustments                 
3: Uprating by Average Adjustment        
4: Extended Policy Simulation            
5: Parental leave benefits               
6: Minimum Wage Adjustments              
7: HHoT unemployment extension           
8: EUROMOD JRC-Interface                 
9: HHoT - Extended Simulation            
10: HHoT - Non Compulsory Payments       
11: Benefit Calibration Adjustments      
12: Consumption Inflation Adjustment     
13: HHoT - Monthly Unemployment extension
14: HFCS data                            
15: HHoT - childcare costs               
16: HHoT - sickness benefits             

(navigate)=
## 3. Navigating the model

The connector mirrors the hierarchy of the EUROMOD user interface as a tree of
Python objects. Containers can be indexed **by position** or **by name**, and
they slice like lists.

In [5]:
mod.countries          # a Container of Country objects

0: AT 
1: BE 
2: BG 
3: CY 
4: CZ 
5: DE 
6: DK 
7: EE 
8: EL 
9: ES 
10: FI
11: FR
12: HR
13: HU
14: IE
15: IT
16: LT
17: LU
18: LV
19: MT
20: NL
21: PL
22: PT
23: RO
24: SE
25: SI
26: SK
27: SL

In [6]:
mod.countries["IT"]    # index by two-letter country code

------------------------------
Country
------------------------------
	 ct_factors: 434 elements
	 datasets: 42 elements
	 extensions: 17 elements
	 local_extensions: 0 elements
	 name: 'IT'
	 policies: 57 elements
	 systems: 21 elements
	 upratefactors: 46 elements

A `Country` groups everything defined for that country: its `datasets`,
`policies`, `systems`, and any country-local `extensions`. As with the model,
printing shows the shape of each.

In [7]:
it = mod.countries["IT"]
it

------------------------------
Country
------------------------------
	 ct_factors: 434 elements
	 datasets: 42 elements
	 extensions: 17 elements
	 local_extensions: 0 elements
	 name: 'IT'
	 policies: 57 elements
	 systems: 21 elements
	 upratefactors: 46 elements

(spine)=
## 4. The spine: policies, functions, parameters

The **spine** is EUROMOD's ordered sequence of tax-benefit calculations. It has
three nested levels:

```
Policy   (e.g. income tax)
  └─ Function  (e.g. a benefit calculation, an eligibility test)
       └─ Parameter  (e.g. a rate, a threshold, a formula)
```

These are **defined** at the `Country` level (the design of the rules) and
**implemented** at the `System` level (the design *as parameterised for a given
policy year*). That distinction matters: the same policy has different parameter
values in `IT_2019` and `IT_2020`.

Let's look at the policies defined for Italy.

In [8]:
it.policies

0: SetDefault_it             |                                            |    DEF: DEFAULT VALUES 
1: Uprate_it                 |                                            |    DEF: UPRATING FACTORS 
2: uprate_bands_it           |  (with switch set for UAA)                 |    DEF: Uprating in bands 
3: ConstDef_it               |                                            |    DEF: CONSTANTS AND  INITIALISATION OF VARIABLES 
4: random_it                 |                                            |    DEF: assign random numbers 
5: ILsDef_it                 |                                            |    DEF: STANDARD INCOME CONCEPTS 
6: ILsUDBDef_it              |                                            |    DEF: UDB related STANDARD INCOME LISTS 
7: ILDef_it                  |                                            |    DEF: INCOME CONCEPTS 
8: TransLMA_it               |                                            |    DEF: Modelling labour market transitions (DO NOT S 

Drill into a single policy to see its functions, and into a function to see its
parameters. Here we inspect the personal income tax policy `tinrt_it`.

> **Indexing note.** Country and system containers can be indexed by name
> (`mod.countries["IT"]`), but `policies` and `functions` are keyed by their
> internal ID, so we pick one out by its readable name with a simple
> comprehension.

In [9]:
tax = [p for p in it.policies if p.name == "tinrt_it"][0]
print(tax.functions)
print(tax.functions[0].parameters)

0: DefConst    |     
1: ArithOp     |     
2: ArithOp     |    Proportion of rental income which is taxable 
3: ArithOp     |     
4: ArithOp     |    Tax on rental income 
5: ArithOp     |    Tax on rental income 



0: $tinrt_IncomeTaxable    |    taxable rental income 
1: $tinrt_FixedRate        |    Rental income is subject to a fixed rate: 21% 
2: $tinrt_FixedRate1       |    Rental income is subject to a fixed rate: 10% 
3: [Placeholder]           |     
4: [Placeholder]           |     
5: [Placeholder]           |     



The **system-level** view adds the state that applies for a given year — most
importantly the `switch` (whether a policy/function is active) and the concrete
parameter `value`s. Compare the country-level definition above with the
`IT_2020` implementation below: the system view shows each policy's on/off
switch.

In [10]:
it.systems["IT_2020"].policies

0: SetDefault_it             | on                                           |    DEF: DEFAULT VALUES 
1: Uprate_it                 | on                                           |    DEF: UPRATING FACTORS 
2: uprate_bands_it           | on (with switch set for UAA)                 |    DEF: Uprating in bands 
3: ConstDef_it               | on                                           |    DEF: CONSTANTS AND  INITIALISATION OF VARIABLES 
4: random_it                 | on                                           |    DEF: assign random numbers 
5: ILsDef_it                 | on                                           |    DEF: STANDARD INCOME CONCEPTS 
6: ILsUDBDef_it              | on                                           |    DEF: UDB related STANDARD INCOME LISTS 
7: ILDef_it                  | on                                           |    DEF: INCOME CONCEPTS 
8: TransLMA_it               | off                                          |    DEF: Modelling labour market tran

(systems)=
## 5. Systems and datasets

A `System` is a country's tax-benefit rules for one policy year. To run one you
must pair it with a **dataset**. A system lists the datasets configured for it,
and marks the **best match** — the dataset the model authors recommend for that
system.

In [11]:
it.systems["IT_2020"].datasets

0: training_data             |             
1: IT_2018_a0                |             
2: IT_2018_a3                |             
3: IT_2019_a0                |             
4: IT_2019_a1                |             
5: IT_2020_hhot              |             
6: IT_2010_a6_2010_03_e1     |             
7: IT_2021_b1                | best match  
8: IT_training_data          |             
9: IT_2021_h1_imp1           |             

The `dataset_id` you pass to `run()` selects which dataset-specific
configuration (uprating factors, default values, currency) EUROMOD applies. It
must be one of the datasets configured for the system. Here we use the training
dataset `IT_training_data`.

(running)=
## 6. Running a simulation

`run()` takes the input microdata as a `pandas.DataFrame` plus the `dataset_id`,
and returns a `Simulation`. String variables are ignored (EUROMOD works on
numeric variables), and the input must contain the `idhh` and `idperson`
identifiers.

In [12]:
data = pd.read_csv(os.path.join(DATA_DIR, "IT_training_data.txt"), sep="\t")
baseline = it.systems["IT_2020"].run(data, "IT_training_data")
baseline

Simulation for system IT_2020 with dataset IT_training_data finished.


------------------------------
Simulation
------------------------------
	 constantsToOverwrite: {}
	 errors: []
	 output_filenames: ['it_2020_std.txt']
	 outputs: Pandas DataFrame of 579 variables and 7482 observations.

The `Simulation` object bundles:

- `outputs` — a container of result DataFrames (most systems produce one; some
  produce several, e.g. when an add-on adds its own output);
- `output_filenames` — the name of each output;
- `errors` — any errors and warnings raised by the engine;
- `constantsToOverwrite` — a record of the constants you changed (see below).

Access the main output by position or by filename:

In [13]:
baseline.outputs[0].head()

,idhh,idperson,idmother,idfather,...,tu_it_it_IsPartner,tu_it_it_IsOwnDependentChild,tu_it_it_IsDepParent,tu_fa_family_it_HeadID
0,1.0,101.0,0.0,0.0,...,0.0,0.0,0.0,101.0
1,2.0,201.0,0.0,0.0,...,0.0,0.0,0.0,201.0
2,3.0,301.0,0.0,0.0,...,0.0,0.0,0.0,301.0
3,4.0,401.0,0.0,0.0,...,0.0,0.0,0.0,401.0
4,5.0,501.0,0.0,0.0,...,0.0,0.0,0.0,501.0


By default `run()` prints a completion message and any warnings. Pass
`verbose=False` to silence that (useful in loops), and check `sim.errors`
yourself. If the simulation aborts, `run()` raises an exception.

(switches)=
## 7. Building reforms I — policy switches

The real power of the connector is running **counterfactuals**. The simplest
reform switches a policy on or off. Each `PolicyInSystem` has a `switch`
attribute (`'on'`/`'off'`) you can set before running. Changes live only for the
current Python session — they are **not** written back to the model files.

Below we turn a benefit policy off and measure the effect on mean household
disposable income (`ils_dispy`).

In [14]:
# baseline mean disposable income
base_dispy = baseline.outputs[0].ils_dispy.mean()

# switch a policy off and re-run
bfacc = [p for p in it.systems["IT_2020"].policies if p.name == "bfacc_it"][0]
bfacc.switch = "off"
reform = it.systems["IT_2020"].run(data, "IT_training_data", verbose=False)
reform_dispy = reform.outputs[0].ils_dispy.mean()

# restore the switch so later cells start from the baseline again
bfacc.switch = "on"

print(f"baseline mean disposable income: {base_dispy:,.2f}")
print(f"reform   mean disposable income: {reform_dispy:,.2f}")
print(f"difference:                      {reform_dispy - base_dispy:,.2f}")

baseline mean disposable income: 1,084.51
reform   mean disposable income: 1,084.22
difference:                      -0.29


The same pattern works at the `Function` level (each function-in-system also has
a `switch`). This is the recommended way to toggle components, because it is
explicit and easy to reset.

(constants)=
## 8. Building reforms II — overwriting constants

Many EUROMOD parameters are expressed in terms of named **constants** (e.g.
uprating indices, thresholds). Rather than editing the model, you can override
constants for a single run with the `constantsToOverwrite` argument.

It is a `dict` keyed by a `(constant_name, group)` tuple, where `group` is the
constant's group number as a string (or `""` when it has none), mapping to the
new value as a string.

In [15]:
sim_const = it.systems["IT_2020"].run(
    data, "IT_training_data",
    constantsToOverwrite={("$penIndexA_2007", ""): "1.05"},
    verbose=False,
)
sim_const.constantsToOverwrite

{('$penIndexA_2007', ''): '1.05'}

`sim.constantsToOverwrite` echoes back exactly what was overridden, so a
`Simulation` is self-documenting about the reform it represents.

(addons)=
## 9. Add-ons

**Add-ons** are self-contained blocks of policy logic (e.g. marginal tax rate
calculations, labour-market adjustments) that are woven into the spine at run
time. Pass them via the `addons` argument.

Each add-on is applied for one of its **add-on systems**. You can specify this
two ways:

- **Name only** — `addons=["MTR"]`. EUROMOD resolves the applicable add-on
  system automatically, exactly as the user interface does, by matching the
  run's system against the add-on's applicability rules.
- **Name and system explicitly** — `addons=[("MTR", "MTR")]`, when you want to
  pin a specific add-on system.

The name-only form is the convenient default; reach for the explicit tuple only
when the automatic choice is ambiguous.

In [16]:
# name-only: the add-on system is resolved automatically
mtr = it.systems["IT_2020"].run(data, "IT_training_data", addons=["MTR"], verbose=False)
mtr.output_filenames

['it_2020_base_mtr.txt', 'it_2020_mtr.txt']

The MTR add-on emits its own output alongside the standard one. Here is the mean
marginal tax rate from its dedicated output file:

In [17]:
mtr.outputs["it_2020_mtr.txt"].mtrpc.mean()

20.76796201407752

The explicit form produces the same result; it just names the add-on system
directly instead of relying on resolution:

```python
it.systems["IT_2020"].run(data, "IT_training_data", addons=[("MTR", "MTR")])
```

If the name-only form is **ambiguous** — i.e. more than one add-on system
applies to the run — EUROMOD raises an error asking you to disambiguate. In that
case, supply the system explicitly with the tuple form.

### Inspecting add-ons as objects

Beyond passing them to `run()`, add-ons are also exposed as first-class
objects on the model, so you can explore their structure the same way you
navigate a country. `mod.addons` is a container of `Addon` objects.

In [ ]:
mod.addons

0: CT_XBASE
1: CT_XCES 
2: CT_XCIS 
3: CT_XCQ  
4: LMA     
5: MTR     
6: NRR     
7: TCA     


An `Addon` is navigable like a `Country`: it has `systems` (of type
`AddonSystem`) and a `policies` spine (policies -> functions -> parameters).
Unlike a country it has no datasets of its own, and it simply references the
model-wide `extensions`.

In [ ]:
ao = mod.addons["MTR"]
ao

------------------------------
Addon
------------------------------
	 extensions: 17 elements
	 name: 'MTR'
	 policies: ao_control_MTR, MTR_PREP, MTR_INIT, MTR_STORE, MTR_CALC
	 systems: 26 elements


In [ ]:
ao.systems

0: MTR    
1: MTR_PL 
2: MTR_EE 
3: MTR_SI 
4: MTR_SK 
5: MTR_NL 
6: MTR_RO 
7: MTR_FI 
8: MTR_BG 
9: MTR_DE 
10: MTR_LT
11: MTR_AT
12: MTR_PT
13: MTR_EL
14: MTR_BE
15: MTR_FR
16: MTR_IE
17: MTR_IT
18: MTR_LV
19: MTR_MT
20: MTR_HR
21: MTR_CY
22: MTR_DK
23: MTR_ES
24: MTR_CZ
25: MTR_LU


Each **add-on system** declares which base-country systems it applies to,
using the same wildcard rules as the user interface.
`get_applicable_systems()` returns the add-on systems that match a given base
system - exactly the resolution that `run(addons=["MTR"])` performs for you.

In [ ]:
ao.get_applicable_systems("IT_2020")

0: MTR_IT


In [ ]:
mtr_it = ao.systems["MTR_IT"]
mtr_it.applies_to_patterns, mtr_it.not_applicable_patterns

(['it_20*'], [])

In [ ]:
mtr_it.is_applicable("IT_2020")   # accepts a system name or a System object

True

The add-on's own policies, functions and parameters are navigable too -
handy for seeing exactly what an add-on injects into the spine:

In [ ]:
mtr_calc = [p for p in ao.policies if p.name == "MTR_CALC"][0]
mtr_calc.functions

0: DefVar        |    define new variables 
1: Elig          |    define eligible persons to calculate MTR 
2: ArithOp       |    calculate change in earnings 
3: ArithOp       |    calculate MTR (in percentage terms) 
4: ArithOp       |    calculate the contribution by components: public p ... 
5: ArithOp       |    calculate the contribution by components: means-te ... 
6: ArithOp       |    calculate the contribution by components: non mean ... 
7: ArithOp       |    calculate the contribution by components: taxes 
8: ArithOp       |    calculate the contribution by components: employee ... 
9: ArithOp       |    calculate the contribution by components: self-emp ... 
10: ArithOp      |    calculate the contribution by components: other SI ... 
11: Restore      |    RESTORE results of mtr_baseline tax-benefit calcul ... 
12: DefOutput    |    DEFINE output for marginal tax rate calculations 


> **These objects are a read-only inspection view.** Navigating or editing
> an `Addon` - its systems, policies or parameters - does **not** change what
> a simulation does: at run time EUROMOD reads the add-on afresh from the
> model files. To apply an add-on in a run use the `addons=` argument shown
> above; to change an add-on's contents, edit it in the EUROMOD user
> interface.

(extensions)=
## 10. Extensions

**Extensions** are toggleable variants of the policy logic (e.g. a benefit
take-up adjustment). Unlike add-ons, they are already part of the model; you
switch them on or off per run with the `switches` argument — a list of
`(extension_short_name, on/off)` tuples.

In [18]:
mod.extensions   # available extension short names (model-wide)

0: Benefit Take-up Adjustments           
1: Tax Compliance Adjustments            
2: Full Year Adjustments                 
3: Uprating by Average Adjustment        
4: Extended Policy Simulation            
5: Parental leave benefits               
6: Minimum Wage Adjustments              
7: HHoT unemployment extension           
8: EUROMOD JRC-Interface                 
9: HHoT - Extended Simulation            
10: HHoT - Non Compulsory Payments       
11: Benefit Calibration Adjustments      
12: Consumption Inflation Adjustment     
13: HHoT - Monthly Unemployment extension
14: HFCS data                            
15: HHoT - childcare costs               
16: HHoT - sickness benefits             

In [19]:
# switch on the Benefit Take-up Adjustment (BTA) extension
bta = it.systems["IT_2020"].run(
    data, "IT_training_data",
    switches=[("BTA", True)],
    verbose=False,
)
bta.outputs[0].ils_ben.mean() - baseline.outputs[0].ils_ben.mean()

-0.09019424396342401

A system also exposes its default extension state via
`get_default_extensions(dataset)`, useful for inspecting what is on before you
override anything.

(requested)=
## 11. Requesting specific outputs

By default EUROMOD returns the standard output defined by the model. If you only
need a handful of variables or income lists, request them explicitly. This
produces an additional compact `custom_output` dataset and is much cheaper to
move around than the full output.

In [20]:
sim_custom = it.systems["IT_2020"].run(
    data, "IT_training_data",
    requested_vars=["ils_dispy", "ils_origy"],
    verbose=False,
)
[df.shape for df in sim_custom.outputs]

[(7482, 579), (7482, 3)]

Related arguments let you request income lists (`requested_incomelists`),
variable groups (`requested_vargroups`), and income-list groups
(`requested_ilgroups`).

(options)=
## 12. Run options reference

A quick reference of the most useful `run()` arguments (see the API reference
for the full signature):

| Argument | Purpose |
| --- | --- |
| `data`, `dataset_id` | **Required.** Input microdata and the dataset config to apply. |
| `constantsToOverwrite` | Override named constants for this run. |
| `addons` | Add-ons to weave in — name-only or `(name, system)`. |
| `switches` | Extensions to switch on/off — `(short_name, bool)`. |
| `requested_vars` / `requested_incomelists` | Return a compact custom output. |
| `euro` | Force monetary output in euro. |
| `public_components_only` | Ignore private components of the model. |
| `outputpath` | Also write the output to disk at this path. |
| `verbose` | Print progress/warnings (default `True`). |

(multicountry)=
## 13. Working across countries

Because everything is plain Python objects, comparative work is just a loop.
Here we compute mean disposable income under the latest system of several
countries, each on its own training data.

In [21]:
results = {}
for cc, sysname in [("AT", "AT_2025"), ("IT", "IT_2020"), ("SL", "SL_1996")]:
    d = pd.read_csv(os.path.join(DATA_DIR, f"{cc}_training_data.txt"), sep="\t")
    sim = mod.countries[cc].systems[sysname].run(d, f"{cc}_training_data", verbose=False)
    results[cc] = sim.outputs[0].ils_dispy.mean()

pd.Series(results, name="mean_disposable_income")

AT    1694.418332
IT    1084.507587
SL    1061.144695
Name: mean_disposable_income, dtype: float64

(caveats)=
## 14. Caveats

- **Changes are session-only.** Setting a `switch`, a parameter `value`, or
  passing `constantsToOverwrite` affects only the current run(s); nothing is
  written back to the model files. For permanent changes use the EUROMOD user
  interface.
- **The connector does not validate model edits.** Changing structural
  attributes (e.g. object `ID`s) can produce meaningless results — don't.
- **Numeric input only.** String variables in your DataFrame are not passed to
  the engine; the data must contain `idhh` and `idperson`.
- **Reset what you change.** When toggling switches inside a loop or notebook,
  restore them afterwards (as we did above) so later cells start from a known
  baseline.